<a href="https://colab.research.google.com/github/shiwangiedulearn-jpg/ai-engineer-learning/blob/main/chunking_day4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
medical_documents = [
    """
    High blood pressure, also called hypertension, means that the force
    of blood against artery walls is consistently too high. Persistent
    hypertension can increase the risk of cardiovascular disease.
    """,

    """
    Regular physical activity can support cardiovascular health.
    Adults should generally aim for regular moderate-intensity activity,
    depending on their individual health circumstances.
    """,

    """
    High cholesterol can contribute to plaque buildup in arteries.
    LDL cholesterol is commonly referred to as bad cholesterol because
    elevated levels can increase cardiovascular risk.
    """,

    """
    Diabetes is a condition involving abnormal blood glucose regulation.
    Long-term uncontrolled blood sugar can affect multiple organs.
    """
]

In [2]:
full_text = "\n\n".join(medical_documents)  #\n\n put 2 new lines bw 2 doc
print(full_text)


    High blood pressure, also called hypertension, means that the force
    of blood against artery walls is consistently too high. Persistent
    hypertension can increase the risk of cardiovascular disease.
    


    Regular physical activity can support cardiovascular health.
    Adults should generally aim for regular moderate-intensity activity,
    depending on their individual health circumstances.
    


    High cholesterol can contribute to plaque buildup in arteries.
    LDL cholesterol is commonly referred to as bad cholesterol because
    elevated levels can increase cardiovascular risk.
    


    Diabetes is a condition involving abnormal blood glucose regulation.
    Long-term uncontrolled blood sugar can affect multiple organs.
    


In [28]:
def chunk_text(text, chunk_size= 300, overlap= 50):
  chunks = []
  start=0
  while start< len(text):
    end = start+ chunk_size
    chunks.append(text[start:end])
    start = end - overlap

  return chunks


In [29]:
chunks = chunk_text(full_text)


In [30]:
for i, chunk in enumerate(chunks):
  print(f"CHUNK {i}")
  print(chunk)

CHUNK 0

    High blood pressure, also called hypertension, means that the force
    of blood against artery walls is consistently too high. Persistent
    hypertension can increase the risk of cardiovascular disease.
    


    Regular physical activity can support cardiovascular health.
    Adults should 
CHUNK 1
 support cardiovascular health.
    Adults should generally aim for regular moderate-intensity activity,
    depending on their individual health circumstances.
    


    High cholesterol can contribute to plaque buildup in arteries.
    LDL cholesterol is commonly referred to as bad cholesterol be
CHUNK 2
erol is commonly referred to as bad cholesterol because
    elevated levels can increase cardiovascular risk.
    


    Diabetes is a condition involving abnormal blood glucose regulation.
    Long-term uncontrolled blood sugar can affect multiple organs.
    
CHUNK 3
rgans.
    


In [31]:
from sentence_transformers import SentenceTransformer

In [32]:
model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [33]:
chunk_embeddings = model.encode(chunks)

In [94]:
query = "what can increase cardiovascular risk?"

In [95]:
query_embedding= model.encode([query])

In [96]:
from sklearn.metrics.pairwise import cosine_similarity

In [97]:
scores = cosine_similarity(query_embedding, chunk_embeddings)[0]

In [98]:
print(scores)

[0.50828916 0.56320715 0.41073424 0.11239047]


In [121]:
top_k = 3

In [122]:
top_indices = scores.argsort()[::-1][:top_k]

In [123]:
for index in top_indices:
  print("SCORE:", round(scores[index],4))
  print(chunks[index])

SCORE: 0.5632
 support cardiovascular health.
    Adults should generally aim for regular moderate-intensity activity,
    depending on their individual health circumstances.
    


    High cholesterol can contribute to plaque buildup in arteries.
    LDL cholesterol is commonly referred to as bad cholesterol be
SCORE: 0.5083

    High blood pressure, also called hypertension, means that the force
    of blood against artery walls is consistently too high. Persistent
    hypertension can increase the risk of cardiovascular disease.
    


    Regular physical activity can support cardiovascular health.
    Adults should 
SCORE: 0.4107
erol is commonly referred to as bad cholesterol because
    elevated levels can increase cardiovascular risk.
    


    Diabetes is a condition involving abnormal blood glucose regulation.
    Long-term uncontrolled blood sugar can affect multiple organs.
    


In [124]:
from google import genai

print("google-genai is working!")

google-genai is working!


In [125]:
from google.colab import userdata
GEMINI_API_KEY= userdata.get("GEMINI_API_KEY")
print("API key loaded", GEMINI_API_KEY is not None)

API key loaded True


In [126]:
client = genai.Client(api_key= GEMINI_API_KEY)
print("Gemini client created successfully!")

Gemini client created successfully!


In [127]:
response = client.models.generate_content(
    model="gemini-3.6-flash",
    contents="explain machine learning in one sentence"
)
print(response.text)

**Machine learning** is a branch of artificial intelligence where computers analyze data to identify patterns and make decisions or predictions on their own, continuously improving without being explicitly programmed for every task.


In [128]:
retrieved_chunks = [chunks[i] for i in top_indices]

In [129]:
context = "\n\n".join(retrieved_chunks)

In [130]:
prompt = f"""
You are an assistant that answers questions using the provided context.

Use only the information provided in the context.

If the answer cannnot be found in context, say:
"I dont have enough information in the provided context."
Context:
{context}

Question:
{query}

Answer:
"""

In [131]:
response= client.models.generate_content(
    model="gemini-3.6-flash",
    contents=prompt

)
print(response.text)

Based on the provided context, the following can increase cardiovascular risk:

* **Persistent hypertension** (high blood pressure)
* **Elevated levels of bad cholesterol** (LDL cholesterol)
